# Step 4 — The Same Agent, Built With LangGraph

Frameworks don't do magic: they package the loop you wrote in Step 3.

| Step 3 (by hand) | Step 4 (LangGraph) |
|---|---|
| Tools listed as text in the prompt | `tool(fn)` + `.bind_tools()`: native tool calling (JSON) |
| Regex parses `Action:` / `Action Input:` | The model returns structured `tool_calls` |
| `for step in range(max_steps)` | `StateGraph`: agent node ⇄ tools node |
| `if parsed["final"]: return` | `tools_condition` edge: no tool call → END |
| `run_tool(name, input)` | `ToolNode` runs the requested tool |
| `max_steps` | `recursion_limit` |

### Setup
Run this cell first. It lets the notebook import the lab's own code (`config.py`, `rag/`, `agent/`, `llm.py`) from the folder above `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

In [ ]:
from rag.ingest import build_index

build_index()

In [ ]:
from llm import check_llm

ok, message = check_llm()
print("LLM ready ✅" if ok else f"LLM NOT ready ❌  {message}")

## 1. Look at the graph
The whole agent is two nodes and three edges. Paste the Mermaid text into <https://mermaid.live> to see it drawn.

In [ ]:
from agent.graph_agent import build_graph

print(build_graph().get_graph().draw_mermaid())

## 2. Run the same four questions
`run_graph_agent()` returns the same `{answer, trace}` shape as Step 3, so we can print it the same way.

In [ ]:
DEMO_QUESTIONS = [
    ("What is 18% of 4520, plus 75?", "calculator"),
    ("How many days of Sick Leave can be carried forward?", "search_documents"),
    ("What is the Maternity Benefit Act in India?", "wikipedia_lookup"),
    ("I joined in March. How many Earned Leave days will I have accrued by the end of September, "
     "and can I encash them?", "search_documents, then calculator (7 x 1.5 = 10.5)"),
]

In [ ]:
from agent import print_trace
from agent.graph_agent import run_graph_agent

for question, expected in DEMO_QUESTIONS:
    print("=" * 80)
    print("Q:", question, f"   (expected: {expected})")
    print_trace(run_graph_agent(question))

## Discuss
- The traces look the same as Step 3. What did LangGraph do for us, and what did it hide?
- **Trade-off:** LangGraph needs a model that supports native tool calling. The Step 3 text format works with any model.